In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from main import run_experiment, save_results
from simlm.config import (
    Config,
    ExperimentConfig,
    LLMConfig,
    GroundConfig,
)
import numpy as np

In [ ]:
experiment_types = ["baseline_cot", "simlm"]
max_iterations = [3,5,7]
few_shots = [0, 1, 2]
models = [
    ("openai", "gpt-3.5-turbo"),
    ("openai", "gpt-4.1-nano"),
    ("google", "gemini-2.0-flash"),
    ("google", "gemini-2.5-flash-preview-04-17"),
    ("google", "gemini-1.5-flash"),
]
flat_grounds = [
    {"type": "flat"}
]
sine_grounds = [{"type": "sine", "amplitude": 0.5, "frequency": 1.0}]
interpolated_grounds = [
    {
        "type": "interpolated",
        "difficulty": 1.0,
        "easy": {"amplitude": 0.15, "frequency": 0.25},
        "hard": {
            "amplitudes": [0.6, 0.15, 0.05],
            "frequencies": [0.9, 2.25, 4.5],
        },
    }
    for difficulty in np.arange(0.0, 1.1, 0.1)
]
grounds = flat_grounds + sine_grounds + interpolated_grounds

In [ ]:
# get the product of experiment types, models, and grounds with
# itertools.product
from itertools import product

configs = []

for experiment_type, (model_service, model_name), ground, max_iteration, few_shot in product(
    experiment_types,
    models,
    grounds,
    max_iterations,
    few_shots
    
):
    config = Config(
        experiment=ExperimentConfig(
            type=experiment_type,
            visualize=False,
            save_results=True,
            max_iterations=max_iteration,
            few_shot=few_shot,
            few_shot_examples_path="results\experiment_results-2520.jsonl"

        ),
        llm=LLMConfig(
            service=model_service,
            model_name=model_name,
            temperature=0.5,
        ),
        ground=GroundConfig(**ground),
    )
    configs.append(config)
    # print(config)
print(len(configs))

In [ ]:
configs = configs[-193:]

In [ ]:
for config in configs:
    print(config)
    runner, results = run_experiment(config)
    save_results(results)

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path("results")
result_file = results_dir / "experiment_results.jsonl"

result_df = pd.read_json(result_file, lines=True)
config_df = pd.json_normalize(result_df["config"])
result_df = pd.concat([result_df, config_df], axis=1)
result_df = result_df.drop(columns=["config"])
result_df